In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings("ignore")

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(


In [2]:
def opening_downscaling_cmip(model, exp, country, month, start, end):
    if exp == 'historical':
        start_year = 1985
        end_year   = 2014
    else:
        start_year = start
        end_year   = end
    downscaled_path = f'../data/downscaled_data/{country}/{exp}/{model}/'
    ds = xr.open_dataset(downscaled_path+f'10m_wind_downscaled_{country}_m{month}_{start_year}_{end_year}.nc')
    return ds

In [3]:
def calculate_future_delta(atlas_path, country, month,
                           hist_ds, ssp_ds, exp, start, end,
                           model="CNRM-ESM2-1"):
    # 1) present-day baseline
    pres_ds = xr.open_dataset(f"{atlas_path}/10m_wind_integrated_{country}_m{month}.nc")

    # 2) make a writable copy that already contains the integrated fields
    future_ds = pres_ds.copy()

    # ---- wds ----------------------------------------------------------
    wds_delta = (ssp_ds["wds_downscaled"] - hist_ds["wds_downscaled"])
    future_ds["wds_delta"]      = wds_delta
    future_ds["wds_integrated"] = pres_ds["wds_integrated"] + wds_delta

    # ---- u10 ----------------------------------------------------------
    u10_delta = (ssp_ds["u10_downscaled"] - hist_ds["u10_downscaled"])
    future_ds["u10_delta"]      = u10_delta
    future_ds["u10_integrated"] = pres_ds["u10_integrated"] + u10_delta

    # ---- v10 ----------------------------------------------------------
    v10_delta = (ssp_ds["v10_downscaled"] - hist_ds["v10_downscaled"])
    future_ds["v10_delta"]      = v10_delta
    future_ds["v10_integrated"] = pres_ds["v10_integrated"] + v10_delta

    # ---- dir ----------------------------------------------------------
    dir_delta = (ssp_ds["dir_downscaled"] - hist_ds["dir_downscaled"])
    future_ds["dir_delta"]      = dir_delta
    future_ds["dir_integrated"] = pres_ds["dir_integrated"] + dir_delta

    # 3) save
    out_dir = f"{atlas_path}/{exp}/{model}"
    os.makedirs(out_dir, exist_ok=True)
    out_name = f"{out_dir}/10m_wind_{country}_m{month}_{exp}_{start}_{end}.nc"
    future_ds.to_netcdf(out_name)

    return future_ds

In [4]:
month = 1
country = 'cuba'
model = 'CNRM-ESM2-1'

start = 2030
end   = 2049

atlas_path = f'../data/atlas_data/{country}/'

In [5]:
hist_ds = opening_downscaling_cmip(model, 'historical', country, month, 1985, 2014)

In [7]:
s370_ds = opening_downscaling_cmip(model, 'ssp370', country, month, start, end)
ssp370_proj = calculate_future_delta(atlas_path, country, month, hist_ds, s370_ds, 
                                   'ssp370', start, end)

In [ ]:
s585_ds = opening_downscaling_cmip(model, 'ssp585', country, month, start, end)
ssp585_proj = calculate_future_delta(atlas_path, country, month, hist_ds, s585_ds, 
                                   'ssp585', start, end)